# 📏 Member 3: Systematic Sampling & Periodicity Analysis
**Project:** Design, Estimation and Efficiency Comparison of Sampling Techniques on IPL Batting Data (2008–2024)

### Objectives:
1. Construct the sorted sampling frame: (date, match_id, inning, batting_position).
2. Justify sampling interval $k = 15$ ($N = 16,515 = 15 \times 1,101$).
3. Draw a 1-in-15 systematic sample and compute single-sample estimators.
4. Enumerate all $k = 15$ possible systematic samples to obtain the exact empirical variance.
5. Diagnose periodicity resulting from $k \approx 15$ matching the natural batting order per match.
6. Conduct One-Way ANOVA across systematic samples.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from sampling.frame import load_frame
from sampling import systematic, estimators as est
from sampling.tests import anova_systematic

frame = load_frame()
print(f"Frame sorted for systematic sampling. N = {len(frame):,}, k = {config.k}, n = {config.n}")


## 1. Single Systematic Sample Draw & Estimation


In [ ]:
true_mean = float(frame["runs"].mean())
true_total = float(frame["runs"].sum())
true_prop = float((frame["runs"] >= 30).mean())

s_sys = systematic.draw_systematic(frame, k=config.k, start=0)
res_sys = systematic.analyse_systematic(s_sys, frame, k=config.k, N=config.N,
                                        true_mean=true_mean, true_total=true_total, true_prop=true_prop)

print(f"Systematic Sample Mean: {res_sys['mean']:.4f}")
print(f"SE (treat-as-SRS): {res_sys['se_mean']:.4f}")
print(f"Variance (successive differences): {res_sys['var_mean_succ_diff']:.6f}")
print(f"95% CI: ({res_sys['ci_mean'][0]:.2f}, {res_sys['ci_mean'][1]:.2f})")
print(f"Absolute Error: {res_sys['abs_error_mean']:.4f}")


## 2. Complete Enumeration across all $k = 15$ Starts
Because systematic sampling has exactly $k$ possible samples in the sample space, we can calculate the **exact population variance** of the estimator:
$$V(\bar{y}_{sys}) = \frac{1}{k} \sum_{r=1}^k (\bar{y}_r - \bar{Y})^2$$


In [ ]:
emp = systematic.empirical_variance_across_starts(frame, k=config.k)

print(f"Exact Systematic Variance V(y_sys): {emp['var_systematic_mean']:.6f}")
print(f"Analytic SRS Variance V(y_SRS):     {emp['var_srs_mean']:.6f}")
print(f"Relative Efficiency vs SRS:         {emp['relative_efficiency_vs_srs']:.4f}")
print(f"\nPeriodicity insight:\n{emp['periodicity_comment']}")

df_starts = pd.DataFrame({
    "start_r": range(config.k),
    "sample_mean": emp["sample_means"],
    "sample_prop": emp["sample_proportions"]
})
df_starts


## 3. ANOVA across Systematic Samples


In [ ]:
sys_anova = anova_systematic(frame, k=config.k)
print(f"Test: {sys_anova['test']}")
print(f"F-value: {sys_anova['F_value']:.4f}, p-value: {sys_anova['p_value']:.4f}")
print(f"Decision: {sys_anova['decision']}")


## 4. Comb Visualisation (The 1-in-15 Pick Pattern)


In [ ]:
show_n = 200
x = np.arange(show_n)
y = frame["runs"].values[:show_n]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(x, y, color="#D3D3D3", width=1.0, label="Population Units")

colors = ["#264653", "#2A9D8F", "#E76F51"]
for i, start in enumerate([0, 5, 10]):
    picks = np.arange(start, show_n, config.k)
    ax.bar(picks, frame["runs"].values[picks], width=1.0, color=colors[i], alpha=0.85, label=f"Start r={start}")

ax.set_title(f"Systematic Sampling Comb (k = {config.k}): Selection Pattern")
ax.set_xlabel("Frame Index")
ax.set_ylabel("Runs")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()
